# NPU vs GPU vs CPU 性能对比实验

## 实验目标
1. 对比 NPU / GPU / CPU 上的训练性能和吞吐量
2. 验证 NPU 和 GPU 上的模型精度一致性


In [ ]:
# 先设置环境变量抑制警告，子进程（spawn）也会继承
import os
os.environ['PYTHONWARNINGS'] = 'ignore'

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision.datasets import ImageFolder
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from copy import deepcopy
import math
import time
import matplotlib.pyplot as plt
import numpy as np

try:
    import torch_npu
    _npu_available = True
except ImportError:
    _npu_available = False


import warnings
warnings.filterwarnings("ignore", message=".*owner does not match.*")
warnings.filterwarnings("ignore", message=".*TASK_QUEUE_ENABLE.*")
warnings.filterwarnings("ignore", message=".*Permission mismatch.*")

In [ ]:
# ====== MobileNetV3 完整模型定义（自包含，不依赖外部文件）======

def get_model_parameters(model):
    total_parameters = 0
    for layer in list(model.parameters()):
        layer_parameter = 1
        for l in list(layer.size()):
            layer_parameter *= l
        total_parameters += layer_parameter
    return total_parameters

def _make_divisible(v, divisor=8, min_value=None):
    if min_value is None:
        min_value = divisor
    new_v = max(min_value, int(v + divisor / 2) // divisor * divisor)
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v

def _weights_init(m):
    if isinstance(m, nn.Conv2d):
        torch.nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            torch.nn.init.zeros_(m.bias)
    elif isinstance(m, nn.BatchNorm2d):
        m.weight.data.fill_(1)
        m.bias.data.zero_()
    elif isinstance(m, nn.Linear):
        n = m.weight.size(1)
        m.weight.data.normal_(0, 0.01)
        m.bias.data.zero_()

class h_sigmoid(nn.Module):
    def __init__(self, inplace=True):
        super().__init__()
        self.inplace = inplace
    def forward(self, x):
        return F.relu6(x + 3., inplace=self.inplace) / 6.

class h_swish(nn.Module):
    def __init__(self, inplace=True):
        super().__init__()
        self.inplace = inplace
    def forward(self, x):
        out = F.relu6(x + 3., self.inplace) / 6.
        return out * x

class SqueezeBlock(nn.Module):
    def __init__(self, exp_size, divide=4):
        super().__init__()
        self.dense = nn.Sequential(
            nn.Linear(exp_size, exp_size // divide),
            nn.ReLU(inplace=True),
            nn.Linear(exp_size // divide, exp_size),
            h_sigmoid()
        )
    def forward(self, x):
        batch, channels, height, width = x.size()
        out = F.avg_pool2d(x, kernel_size=[height, width]).view(batch, -1)
        out = self.dense(out).view(batch, channels, 1, 1)
        return out * x

class MobileBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernal_size, stride, nonLinear, SE, exp_size):
        super().__init__()
        padding = (kernal_size - 1) // 2
        self.use_connect = stride == 1 and in_channels == out_channels
        self.SE = SE
        activation = nn.ReLU if nonLinear == "RE" else h_swish
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, exp_size, 1, 1, 0, bias=False),
            nn.BatchNorm2d(exp_size), activation(inplace=True))
        self.depth_conv = nn.Sequential(
            nn.Conv2d(exp_size, exp_size, kernal_size, stride, padding, groups=exp_size),
            nn.BatchNorm2d(exp_size))
        if self.SE:
            self.squeeze_block = SqueezeBlock(exp_size)
        self.point_conv = nn.Sequential(
            nn.Conv2d(exp_size, out_channels, 1, 1, 0),
            nn.BatchNorm2d(out_channels), activation(inplace=True))
    def forward(self, x):
        out = self.depth_conv(self.conv(x))
        if self.SE:
            out = self.squeeze_block(out)
        out = self.point_conv(out)
        return x + out if self.use_connect else out

class MobileNetV3(nn.Module):
    def __init__(self, model_mode="LARGE", num_classes=1000, multiplier=1.0, dropout_rate=0.0):
        super().__init__()
        self.num_classes = num_classes
        if model_mode == "LARGE":
            layers = [
                [16, 16, 3, 1, "RE", False, 16], [16, 24, 3, 2, "RE", False, 64],
                [24, 24, 3, 1, "RE", False, 72], [24, 40, 5, 2, "RE", True, 72],
                [40, 40, 5, 1, "RE", True, 120], [40, 40, 5, 1, "RE", True, 120],
                [40, 80, 3, 2, "HS", False, 240], [80, 80, 3, 1, "HS", False, 200],
                [80, 80, 3, 1, "HS", False, 184], [80, 80, 3, 1, "HS", False, 184],
                [80, 112, 3, 1, "HS", True, 480], [112, 112, 3, 1, "HS", True, 672],
                [112, 160, 5, 1, "HS", True, 672], [160, 160, 5, 2, "HS", True, 672],
                [160, 160, 5, 1, "HS", True, 960],
            ]
            init_conv_out = _make_divisible(16 * multiplier)
            self.init_conv = nn.Sequential(
                nn.Conv2d(3, init_conv_out, 3, 2, 1), nn.BatchNorm2d(init_conv_out), h_swish())
            self.block = nn.Sequential(*[
                MobileBlock(_make_divisible(ic * multiplier), _make_divisible(oc * multiplier),
                           k, s, nl, se, _make_divisible(exp * multiplier))
                for ic, oc, k, s, nl, se, exp in layers])
            self.out_conv1 = nn.Sequential(
                nn.Conv2d(_make_divisible(160 * multiplier), _make_divisible(960 * multiplier), 1, 1),
                nn.BatchNorm2d(_make_divisible(960 * multiplier)), h_swish())
            self.out_conv2 = nn.Sequential(
                nn.Conv2d(_make_divisible(960 * multiplier), _make_divisible(1280 * multiplier), 1, 1),
                h_swish(), nn.Dropout(dropout_rate),
                nn.Conv2d(_make_divisible(1280 * multiplier), num_classes, 1, 1))
        elif model_mode == "SMALL":
            layers = [
                [16, 16, 3, 2, "RE", True, 16], [16, 24, 3, 2, "RE", False, 72],
                [24, 24, 3, 1, "RE", False, 88], [24, 40, 5, 2, "RE", True, 96],
                [40, 40, 5, 1, "RE", True, 240], [40, 40, 5, 1, "RE", True, 240],
                [40, 48, 5, 1, "HS", True, 120], [48, 48, 5, 1, "HS", True, 144],
                [48, 96, 5, 2, "HS", True, 288], [96, 96, 5, 1, "HS", True, 576],
                [96, 96, 5, 1, "HS", True, 576],
            ]
            init_conv_out = _make_divisible(16 * multiplier)
            self.init_conv = nn.Sequential(
                nn.Conv2d(3, init_conv_out, 3, 2, 1), nn.BatchNorm2d(init_conv_out), h_swish())
            self.block = nn.Sequential(*[
                MobileBlock(_make_divisible(ic * multiplier), _make_divisible(oc * multiplier),
                           k, s, nl, se, _make_divisible(exp * multiplier))
                for ic, oc, k, s, nl, se, exp in layers])
            self.out_conv1 = nn.Sequential(
                nn.Conv2d(_make_divisible(96 * multiplier), _make_divisible(576 * multiplier), 1, 1),
                SqueezeBlock(_make_divisible(576 * multiplier)),
                nn.BatchNorm2d(_make_divisible(576 * multiplier)), h_swish())
            self.out_conv2 = nn.Sequential(
                nn.Conv2d(_make_divisible(576 * multiplier), _make_divisible(1280 * multiplier), 1, 1),
                h_swish(), nn.Dropout(dropout_rate),
                nn.Conv2d(_make_divisible(1280 * multiplier), num_classes, 1, 1))
        self.apply(_weights_init)
    def forward(self, x):
        out = self.block(self.init_conv(x))
        out = self.out_conv1(out)
        b, c, h, w = out.size()
        return self.out_conv2(F.avg_pool2d(out, [h, w])).view(b, -1)

print("✅ MobileNetV3 模型定义加载完成")

---
## 2. 了解 Ascend NPU 与 torch-npu

### 什么是 Ascend NPU？

华为 Ascend 910 / 910B 是面向 AI 训练的高性能 NPU（Neural Processing Unit）。
与 GPU 类似，NPU 通过大量并行计算单元加速深度学习训练和推理。

### torch-npu 的作用

`torch-npu` 是 PyTorch 的 Ascend NPU 适配插件，提供了与 CUDA 几乎一致的 API：

| 操作 | CUDA（GPU） | torch-npu（NPU） |
|------|-------------|-----------------|
| 导入 | `import torch` | `import torch; import torch_npu` |
| 设备指定 | `torch.device('cuda')` | `torch.device('npu:0')` |
| 可用性检查 | `torch.cuda.is_available()` | `torch.npu.is_available()` |
| 设备数量 | `torch.cuda.device_count()` | `torch.npu.device_count()` |
| 同步 | `torch.cuda.synchronize()` | `torch.npu.synchronize()` |
| 数据迁移 | `.to('cuda')` | `.to('npu:0')` |

**核心原则：** 模型定义、优化器、DataLoader 等代码完全不变，只需修改设备字符串。
这就是 PyTorch 设备无关设计的优势。

参考 `00_QuickStart_CTR_DeepFM.ipynb` 中的用法：
```python
import torch_npu
DEVICE = "npu:0"
```

In [ ]:
devices = []
if _npu_available and torch.npu.is_available():
    devices.append(('NPU', torch.device('npu:0')))
# CPU is always available
devices.append(('CPU', torch.device('cpu')))
print('Available devices:', [d[0] for d in devices])


---
## 3. 准备数据和模型

使用 Tiny ImageNet 数据集和 MobileNetV3-Large 模型。

In [ ]:
# ====== 3. 加载数据集 ======
BATCH_SIZE = 256
NUM_CLASSES = 200
DATA_DIR = 'tiny-imagenet-200'
NUM_WORKERS = 8

# 验证集预处理（简化版，无数据增强）
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_dataset = ImageFolder(
    os.path.join(DATA_DIR, 'val'),
    transform=val_transform
)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=False,
    multiprocessing_context='spawn', persistent_workers=True
)

print(f"验证集: {len(val_dataset):,} 张图像 ({len(val_loader)} batches)")
print(f"类别数: {len(val_dataset.classes)}")

# 创建基准模型
model = MobileNetV3(model_mode="LARGE", num_classes=NUM_CLASSES, multiplier=1.0, dropout_rate=0.0)
total_params = sum(p.numel() for p in model.parameters())
print(f"MobileNetV3-Large 参数量: {total_params:,}")

---
## 4. 性能基准测试

在 CPU / GPU / NPU 上分别运行固定次数的前向+反向传播，对比：
1. **单次迭代时间**（毫秒/步）
2. **吞吐量**（images/sec）

> ⚠️ 性能数据受具体硬件型号影响。以下代码会自动选择可用的设备进行对比。

In [5]:
def bench(device, model, inp, lab, warm=10, iters=50):
    model = model.to(device)
    inp, lab = inp.to(device), lab.to(device)
    opt = optim.RMSprop(model.parameters(), lr=0.01, alpha=0.9, eps=1e-3)
    criterion = nn.CrossEntropyLoss()
    for _ in range(warm):
        opt.zero_grad()
        loss = criterion(model(inp), lab)
        loss.backward()
        opt.step()
    # Synchronize for accelerator devices
    if 'npu' in str(device):
        torch.npu.synchronize()
    start = time.time()
    for _ in range(iters):
        opt.zero_grad()
        loss = criterion(model(inp), lab)
        loss.backward()
        opt.step()
    if 'npu' in str(device):
        torch.npu.synchronize()
    t = time.time() - start
    return {'device': str(device), 'ms': t/iters*1000, 'ips': iters*inp.size(0)/t}


In [ ]:
BATCH_SIZE = 256
NUM_CLASSES = 200
DATA_DIR = 'tiny-imagenet-200'
model = MobileNetV3(model_mode='LARGE', num_classes=NUM_CLASSES, multiplier=1.0, dropout_rate=0.0)
dummy_input = torch.randn(BATCH_SIZE, 3, 224, 224)
dummy_labels = torch.randint(0, NUM_CLASSES, (BATCH_SIZE,))
results = []
for name, dev in devices:
    r = bench(dev, deepcopy(model), dummy_input, dummy_labels,
              warm=5, iters=20 if name=='CPU' else 50)
    results.append(r)
    print(f'{name}: {r["ms"]:.1f}ms, {r["ips"]:.0f} img/s')

---
## 5. 性能对比可视化

In [ ]:
names = [r['device'] for r in results]
times = [r['ms'] for r in results]
ips = [r['ips'] for r in results]
colors = ['#e74c3c' if 'CPU' in n else ('#3498db' if 'GPU' in n else '#2ecc71') for n in names]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,5))
ax1.bar(names, times, color=colors)
ax1.set_ylabel('ms/step')
ax2.bar(names, ips, color=colors)
ax2.set_ylabel('images/s')
plt.show()


---
## 6. 不同 Batch Size 下的吞吐量对比

加速设备的优势在大 batch size 下更明显。测试不同 batch size 下的吞吐量。

In [ ]:
batch_sizes = [8, 16, 32, 64, 128]
cpu_ips_list, accel_ips_list = [], []
for bs in batch_sizes:
    d_inp = torch.randn(bs, 3, 224, 224)
    d_lab = torch.randint(0, 200, (bs,))
    accel_dev = [d for d in devices if d[0] != 'CPU']
    if accel_dev:
        r1 = bench(accel_dev[0][1], deepcopy(model), d_inp, d_lab, warm=3, iters=20)
        accel_ips_list.append(r1['ips'])
    r2 = bench(torch.device('cpu'), deepcopy(model), d_inp, d_lab, warm=2, iters=5)
    cpu_ips_list.append(r2['ips'])
plt.plot(batch_sizes, cpu_ips_list, 'ro-', label='CPU')
if accel_ips_list:
    label = accel_dev[0][0]
    plt.plot(batch_sizes, accel_ips_list, 'go-', label=label)
plt.xlabel('Batch Size')
plt.ylabel('images/s')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 7. 精度一致性验证

NPU 和 GPU 使用相同的训练算法，理论上应得到相同的模型精度。
本节用相同的数据和 seed 在 NPU 和 CPU 上分别训练 3 个 epoch，对比验证准确率。

> ⚠️ 由于浮点运算顺序差异，NPU 和 GPU 的结果可能不完全一致，
> 但差异应在可接受范围内（通常 < 0.5%）。

In [ ]:
def train_quick(dev):
    torch.manual_seed(42)
    m = MobileNetV3(model_mode='LARGE', num_classes=200, multiplier=1.0, dropout_rate=0.2).to(dev)
    opt = optim.RMSprop(m.parameters(), lr=0.01, alpha=0.9, eps=1e-3)
    crit = nn.CrossEntropyLoss()
    for data, target in tiny_train_loader:
        data, target = data.to(dev), target.to(dev)
        opt.zero_grad()
        loss = crit(m(data), target)
        loss.backward()
        opt.step()
    m.eval()
    correct = total = 0
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(dev), target.to(dev)
            _, pred = m(data).max(1)
            correct += pred.eq(target).sum().item()
            total += target.size(0)
    return 100.0 * correct / total


---
## 8. 迁移总结与最佳实践

### 将模型从 GPU 迁移到 NPU 的核心步骤

1. **安装 torch-npu**：`pip install torch torch-npu`
2. **导入 torch_npu**：`import torch_npu`（只需导入，不需要修改其他 API）
3. **修改设备字符串**：将 `'cuda'` 替换为 `'npu:0'`，或者使用自动检测
4. **调整 pin_memory**：NPU 不支持 `pin_memory=True`，需要设为 `False`
5. **同步函数**：`torch.cuda.synchronize()` → `torch.npu.synchronize()`

### 迁移清单

<table style="margin-left: 0; margin-right: auto; border-collapse: collapse; border: 1px solid #ddd;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">修改项</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">CUDA 代码</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">NPU 代码</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">导入</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>import torch</code></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>import torch; import torch_npu</code></td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">设备</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>torch.device('cuda')</code></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>torch.device('npu:0')</code></td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">数据迁移</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>.cuda()</code> / <code>.to('cuda')</code></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>.to('npu:0')</code>（不变）</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">模型迁移</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>.cuda()</code> / <code>.to('cuda')</code></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>.to('npu:0')</code>（不变）</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">DataLoader</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>pin_memory=True</code></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>pin_memory=False</code></td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">同步</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>torch.cuda.synchronize()</code></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>torch.npu.synchronize()</code></td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">可用性</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>torch.cuda.is_available()</code></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>torch.npu.is_available()</code></td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">显存</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>torch.cuda.memory_allocated()</code></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>torch.npu.memory_allocated()</code></td>
    </tr>
  </tbody>
</table>

### 自动检测设备函数（推荐）

```python
def get_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    elif hasattr(torch, 'npu') and torch.npu.is_available():
        return torch.device('npu:0')
    else:
        return torch.device('cpu')
```

### 注意事项

1. **CANN 版本匹配**：torch-npu 需要与 CANN 工具包版本匹配
2. **pin_memory**：NPU 不支持 pinned memory
3. **混合精度**：NPU 支持 AMP，但需使用 `torch.npu.amp` 而非 `torch.cuda.amp`
4. **分布式**：NPU 使用 `hccl` 后端替代 `nccl`：`dist.init_process_group(backend='hccl')`
5. **数值精度**：NPU 和 GPU 的浮点运算顺序可能不同，但最终精度差异 &lt; 0.5%

In [ ]:
print('=' * 60)
print('Performance Summary')
print('=' * 60)
for r in results:
    print(f"  {r['device']}: {r['ips']:.0f} img/s, {r['ms']:.1f} ms/step")

cpu   = [r for r in results if r['device'].lower() == 'cpu']
accel = [r for r in results if r['device'].lower() != 'cpu']

if cpu and accel:
    ratio = accel[0]['ips'] / cpu[0]['ips']
    print(f"\nSpeedup: {ratio:.1f}x")

## 课后练习

1. (单选题) GPU 代码迁移到 NPU 时，设备选择最稳妥的写法是？
   - A. device = "npu:0" if torch.npu.is_available() else "cpu"
   - B. device = "cuda:0" if torch.cuda.is_available() else "cpu"
   - C. device = "npu:0" 无条件
   - D. device = "cpu"

2. (单选题) NPU 计时为什么必须在 forward 后调用 synchronize？
   - A. NPU kernel 异步执行，需要等待完成再取时间戳
   - B. 释放显存
   - C. 防止过拟合
   - D. 触发反向传播

3. (多选题) GPU 到 NPU 迁移需要检查？
   - A. import torch_npu
   - B. torch.cuda.* 替换为 torch.npu.*
   - C. device 字符串由 cuda 改为 npu
   - D. 数据/模型/标签全部 to(device)

4. (多选题) NPU 性能对比实验应固定？
   - A. batch_size 与输入 shape
   - B. warmup 与 repeats
   - C. dtype
   - D. 同步时机

5. (判断题) 纯 NPU 环境导入 torch_npu 后，torch.cuda.is_available() 通常仍为 False，设备选择不能依赖 CUDA。

6. (判断题) NPU 不支持 bf16，混合精度训练必须使用 fp16 且无需梯度缩放。

7. (填空题) 查看 NPU 算力利用率与内存占用使用命令 ____；Python 侧分析算子耗时建议使用 ____。

8. (填空题) 推理显存不足时，最直接有效的调整是减小 ____ 或 ____。

9. (简答题) 为什么 NPU 首次推理耗时通常显著高于后续推理？

10. (简答题) 如何确认模型和张量确实位于 NPU 而不是 CPU？

11. (代码设计题) 编写 benchmark_inference(model, x, warmup, repeats)，要求包含同步、计时循环、返回平均耗时与吞吐。

12. (单选题) NPU 利用率低而 CPU 利用率高，通常意味着？
   - A. 数据加载/预处理成为瓶颈
   - B. 算子计算密集
   - C. 显存不足
   - D. 模型太小

13. (多选题) 混合精度迁移中需要注意？
   - A. 权重/激活使用 bf16 或 fp16
   - B. 损失缩放与梯度裁剪配合
   - C. BN 等敏感层可保持 fp32
   - D. 所有 API 都必须改 dtype

14. (判断题) model、input、label 必须全部迁移到同一 device 才能正常前向与 loss 计算。

15. (简答题) 如何使用 profiler 定位 NPU 空闲等待数据加载的问题？请给出观察指标与改进方案。

> 参考答案见 answer/02.08_npu_migration_answer.ipynb。